# SHL Assessment Recommendation Engine - Debug & Testing Notebook
This notebook walks through the full pipeline step-by-step:
1. Inspect scraped data (FAISS stored documents)
2. See how embeddings/text representations look
3. Query → retrieval → see which chunks FAISS returns
4. Re-ranking → final answer
5. Evaluate on training data with Recall@10

In [1]:
import json
import re
import numpy as np
import pandas as pd
from collections import defaultdict
import openpyxl
from rank_bm25 import BM25Okapi

import config
from embeddings import load_index, build_text_representation, get_embeddings, embed_query, get_embeddings_model

print(f"OpenAI API Key loaded: {config.OPENAI_API_KEY[:10]}...")
print(f"Config: TOP_K_PER_QUERY={config.TOP_K_PER_QUERY}, TOP_K_TO_LLM={config.TOP_K_TO_LLM}, TOP_K_FINAL={config.TOP_K_FINAL}")

c:\Users\A8000\Downloads\Ashish_Genai_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI API Key loaded: sk-proj-4u...
Config: TOP_K_PER_QUERY=40, TOP_K_TO_LLM=70, TOP_K_FINAL=10


## 1. Inspect Scraped Data (What's stored in FAISS)

In [2]:
# Load the raw scraped assessments
with open(config.ASSESSMENTS_FILE, 'r', encoding='utf-8') as f:
    assessments = json.load(f)

print(f"Total assessments scraped: {len(assessments)}")
print(f"Individual: {sum(1 for a in assessments if a.get('category') == 'Individual Test Solutions')}")
print(f"Pre-packaged: {sum(1 for a in assessments if a.get('category') == 'Pre-packaged Job Solutions')}")
print(f"With description: {sum(1 for a in assessments if a.get('description'))}")
print(f"With duration: {sum(1 for a in assessments if a.get('duration_minutes'))}")

Total assessments scraped: 518
Individual: 377
Pre-packaged: 141
With description: 518
With duration: 432


In [3]:
# Show 5 sample assessments as a table
sample_df = pd.DataFrame(assessments[:5])
sample_df[['name', 'url', 'description', 'test_types', 'duration_minutes', 'remote_testing', 'category']]

,name,url,description,test_types,duration_minutes,remote_testing,category
0,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,This report is designed to be given to individ...,"[Ability & Aptitude, Assessment Exercises, Bio...",NaN,True,Individual Test Solutions
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,The.NET Framework 4.5 test measures knowledge ...,[Knowledge & Skills],30.0,True,Individual Test Solutions
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],17.0,True,Individual Test Solutions
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],5.0,True,Individual Test Solutions
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],11.0,True,Individual Test Solutions


## 2. Inspect FAISS Index & Text Representations (Chunks)

In [4]:
# Load the FAISS index and metadata
index, stored_assessments, stored_texts = load_index()

print(f"FAISS index size: {index.ntotal} vectors")
print(f"Vector dimension: {index.d}")
print(f"Stored assessments: {len(stored_assessments)}")
print(f"Stored text chunks: {len(stored_texts)}")

FAISS index size: 518 vectors
Vector dimension: 3072
Stored assessments: 518
Stored text chunks: 518


In [5]:
# Show what each "chunk" looks like — this is the text that gets embedded
print("=" * 80)
print("SAMPLE CHUNKS (text representations stored in FAISS)")
print("=" * 80)
for i in [0, 50, 100, 200, 300]:
    if i < len(stored_texts):
        print(f"\n--- Chunk {i}: {stored_assessments[i]['name']} ---")
        print(stored_texts[i])
        print(f"   [Char length: {len(stored_texts[i])}]")

SAMPLE CHUNKS (text representations stored in FAISS)

--- Chunk 0: Global Skills Development Report ---
Global Skills Development Report is a Ability & Aptitude, Assessment Exercises, Biodata & Situational Judgement, Competencies, Development & 360, Personality & Behaviour assessment. This report is designed to be given to individuals who have completed the Global Skills Assessment (GSA). With coverage across the Great 8 Domains, this… Suitable for Director, Entry-Level, Executive, General Population, Graduate, Manager, Mid-Professional, Front Line Manager, Supervisor roles.
   [Char length: 477]

--- Chunk 50: Business Communications ---
Business Communications is a Knowledge & Skills assessment. This test measures the candidate's knowledge of communicating in the workplace.  It measures the skills necessary to communicate effectively with coworkers at… Takes 35 minutes to complete.
   [Char length: 249]

--- Chunk 100: Entry Level Customer Serv-Retail & Contact Center ---
Entry Level

In [6]:
# Distribution of chunk lengths
lengths = [len(t) for t in stored_texts]
print(f"Chunk length stats:")
print(f"  Min: {min(lengths)} chars")
print(f"  Max: {max(lengths)} chars")
print(f"  Mean: {np.mean(lengths):.0f} chars")
print(f"  Median: {np.median(lengths):.0f} chars")
print(f"\nAll chunks fit in one embedding (no splitting needed): {max(lengths) < 8000}")

Chunk length stats:
  Min: 102 chars
  Max: 477 chars
  Mean: 328 chars
  Median: 326 chars

All chunks fit in one embedding (no splitting needed): True


## 4. Full Pipeline Debug — Query Analyzer → Retriever → Reranker

In [7]:
from graph import query_analyzer_node, retriever_node, reranker_node, GraphState

def debug_full_pipeline(query: str):
    """Run each pipeline node separately and show intermediate outputs."""
    
    # Initial state
    state = GraphState(
        query=query, search_queries=[], skills=[],
        max_duration=None, domain="",
        candidates=[], recommendations=[]
    )
    
    # --- Node 1: Query Analyzer ---
    print("=" * 80)
    print("NODE 1: QUERY ANALYZER")
    print("=" * 80)
    analyzer_output = query_analyzer_node(state)
    state.update(analyzer_output)
    
    print(f"Generated search queries:")
    for i, sq in enumerate(state['search_queries'], 1):
        print(f"  {i}. {sq}")
    print(f"\nExtracted skills: {state['skills']}")
    print(f"Max duration: {state['max_duration']}")
    print(f"Domain: {state['domain']}")
    
    # --- Node 2: Retriever ---
    print(f"\n{'=' * 80}")
    print(f"NODE 2: HYBRID RETRIEVER (FAISS + BM25, top_k_per_query={config.TOP_K_PER_QUERY}, sending top {config.TOP_K_TO_LLM} to LLM)")
    print(f"  Fusion: 0.6 * FAISS (semantic) + 0.4 * BM25 (keyword)")
    print("=" * 80)
    retriever_output = retriever_node(state)
    state.update(retriever_output)
    
    print(f"Retrieved {len(state['candidates'])} candidates")
    print(f"\n{'Rank':<5} {'Score':<8} {'Name':<45} {'Test Types':<25}")
    print("-" * 85)
    for i, c in enumerate(state['candidates'][:25], 1):
        types = ', '.join(c.get('test_type', [])[:2])
        print(f"{i:<5} {c['score']:<8.4f} {c['name'][:44]:<45} {types:<25}")
    
    # --- Node 3: Reranker ---
    print(f"\n{'=' * 80}")
    print(f"NODE 3: RERANKER (LLM re-ranking, selecting top {config.TOP_K_FINAL})")
    print("=" * 80)
    reranker_output = reranker_node(state)
    state.update(reranker_output)
    
    print(f"\nFinal {len(state['recommendations'])} recommendations:")
    print(f"\n{'#':<4} {'Name':<45} {'Test Types':<30} {'Duration':<10} {'URL'}")
    print("-" * 130)
    for i, r in enumerate(state['recommendations'], 1):
        types = ', '.join(r.get('test_type', []))
        dur = str(r.get('duration') or '-')
        print(f"{i:<4} {r['name'][:44]:<45} {types[:29]:<30} {dur:<10} {r['url']}")
    
    return state

In [8]:
# Full pipeline debug for a sample query
state = debug_full_pipeline("I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.")

NODE 1: QUERY ANALYZER
Generated search queries:
  1. Core Java entry level assessment
  2. Core Java advanced level
  3. Java 8 programming test
  4. Automata coding simulation
  5. Automata Fix code debugging
  6. technology professional job focused assessment
  7. professional 8.0 JFA
  8. agile software development
  9. object oriented programming
  10. collaboration and teamwork assessment
  11. interpersonal communications skills
  12. software development knowledge test
  13. Automata Java code simulation
  14. Java problem solving assessment
  15. professional 7.1 solution
  16. team effectiveness questionnaire
  17. communication skills for developers
  18. Java developer scenario-based simulation
  19. I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.

Extracted skills: ['Java', 'collaboration', 'object oriented programming', 'agile', 'communication', 'problem solving'

In [ ]:
# Another pipeline debug
state = debug_full_pipeline("I am looking for a COO for my company in China and I want to see if they are culturally a right fit for our company. Suggest me an assessment that they can complete in about an hour")

## 5. Evaluate on Training Data — Recall@10 per query

In [9]:
from evaluate import normalize_url, load_train_set, compute_recall_at_k
from graph import recommend

# Load training data
dataset_path = 'input/Gen_AI Dataset.xlsx'
train_data = load_train_set(dataset_path)

print(f"Training queries: {len(train_data)}")
print(f"Total labeled URLs: {sum(len(v) for v in train_data.values())}")
print()
for i, (q, urls) in enumerate(train_data.items(), 1):
    print(f"  Q{i} ({len(urls)} labels): {q[:80]}...")

Training queries: 10
Total labeled URLs: 65

  Q1 (5 labels): I am hiring for Java developers who can also collaborate effectively with my bus...
  Q2 (9 labels): I want to hire new graduates for a sales role in my company, the budget is for a...
  Q3 (6 labels): I am looking for a COO for my company in China and I want to see if they are cul...
  Q4 (5 labels): KEY RESPONSIBITILES:

Manage the sound-scape of the station through appropriate ...
  Q5 (5 labels): Content Writer required, expert in English and SEO....
  Q6 (9 labels): Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
  Q7 (6 labels): ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
  Q8 (5 labels): We're looking for a Marketing Manager who can drive Recro’s brand positioning, c...
  Q9 (5 labels): Based on the JD below recommend me assessment for the Consultant position in my ...
  Q10 (10 labels): I want to hire a Senior Data Analyst with 5 years of exp

In [10]:
# Run evaluation on all training queries
results_table = []
K = config.TOP_K_FINAL  # 15

for i, (query, relevant_urls) in enumerate(train_data.items(), 1):
    recommendations = recommend(query)
    rec_urls = [r['url'] for r in recommendations]
    
    recall = compute_recall_at_k(rec_urls, relevant_urls, k=K)
    
    # Count hits/misses
    rec_normalized = [normalize_url(u) for u in rec_urls]
    hits = []
    misses = []
    for url in relevant_urls:
        name = url.split('/view/')[-1].rstrip('/')
        if normalize_url(url) in rec_normalized:
            hits.append(name)
        else:
            misses.append(name)
    
    results_table.append({
        'Query': f"Q{i}: {query[:60]}...",
        'Relevant': len(relevant_urls),
        'Hits': len(hits),
        'Misses': len(misses),
        f'Recall@{K}': f"{recall:.2f}",
        'Hit Names': ', '.join(hits[:3]),
        'Miss Names': ', '.join(misses[:3]),
    })
    print(f"Q{i}: Recall@{K}={recall:.2f} | Hits={len(hits)}/{len(relevant_urls)}")

mean_recall = np.mean([float(r[f'Recall@{K}']) for r in results_table])
print(f"\n>>> Mean Recall@{K} = {mean_recall:.4f} <<<")

Q1: Recall@10=0.80 | Hits=4/5
Q2: Recall@10=0.56 | Hits=5/9
Q3: Recall@10=0.83 | Hits=5/6
Q4: Recall@10=0.40 | Hits=2/5
Q5: Recall@10=0.20 | Hits=1/5
Q6: Recall@10=0.67 | Hits=6/9
Q7: Recall@10=0.83 | Hits=5/6
Q8: Recall@10=0.80 | Hits=4/5
Q9: Recall@10=0.80 | Hits=4/5
Q10: Recall@10=0.50 | Hits=5/10

>>> Mean Recall@10 = 0.6390 <<<


In [ ]:
# Show results as a DataFrame
results_df = pd.DataFrame(results_table)
results_df

,Query,Relevant,Hits,Misses,Recall@10,Hit Names,Miss Names
0,Q1: I am hiring for Java developers who can al...,5,5,0,1.00,"automata-fix-new, core-java-entry-level-new, j...",
1,Q2: I want to hire new graduates for a sales r...,9,2,7,0.22,"entry-level-sales-solution, interpersonal-comm...","entry-level-sales-7-1, entry-level-sales-sift-..."
2,Q3: I am looking for a COO for my company in C...,6,3,3,0.50,"occupational-personality-questionnaire-opq32r,...","enterprise-leadership-report, opq-team-types-a..."
3,Q4: KEY RESPONSIBITILES:\n\nManage the sound-s...,5,3,2,0.60,"shl-verify-interactive-inductive-reasoning, ma...","verify-verbal-ability-next-generation, english..."
4,"Q5: Content Writer required, expert in English...",5,4,1,0.80,"english-comprehension-new, written-english-v1,...",drupal-new
5,Q6: Find me 1 hour long assesment for the belo...,9,3,6,0.33,"htmlcss-new, sql-server-new, manual-testing-new","automata-selenium, professional-7-1-solution, ..."
6,"Q7: ICICI Bank Assistant Admin, Experience req...",6,2,4,0.33,"administrative-professional-short-form, verify...","financial-professional-short-form, bank-admini..."
7,Q8: We're looking for a Marketing Manager who ...,5,0,5,0.00,,"manager-8-0-jfa-4310, microsoft-excel-365-esse..."
8,Q9: Based on the JD below recommend me assessm...,5,2,3,0.40,"verify-verbal-ability-next-generation, occupat...","shl-verify-interactive-numerical-calculation, ..."
9,Q10: I want to hire a Senior Data Analyst with...,10,3,7,0.30,"sql-server-new, automata-sql-new, python-new",sql-server-analysis-services-%28ssas%29-%28new...


In [ ]:
# Deep dive into the worst performing query — see what went wrong
worst_idx = np.argmin([float(r['Recall@10']) for r in results_table])
worst_query = list(train_data.keys())[worst_idx]
worst_urls = train_data[worst_query]

print(f"WORST QUERY (Q{worst_idx+1}): {worst_query[:100]}...")
print(f"\nExpected URLs ({len(worst_urls)}):")
for u in worst_urls:
    name = u.split('/view/')[-1].rstrip('/')
    print(f"  - {name}")

print(f"\n--- Running full debug pipeline ---")
state = debug_full_pipeline(worst_query)

## 6. Embedding Similarity Analysis

In [ ]:
# Check: are the expected assessments even close in embedding space?
# For the worst query, embed it and check distance to expected assessments
from evaluate import normalize_url as norm_url_check

q_emb = embed_query(worst_query[:500])
q_norm = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)

print("Cosine similarity of EXPECTED assessments to the query:")
print(f"{'Assessment':<50} {'Similarity':<10} {'In Top-40?'}")
print("-" * 75)

# Get top-40 indices for comparison
_, top40_indices = index.search(q_norm, 40)
top40_set = set(top40_indices[0].tolist())

for expected_url in worst_urls:
    # Find this URL in our assessments
    found = False
    for idx, a in enumerate(stored_assessments):
        if norm_url_check(a['url']) == norm_url_check(expected_url):
            # Reconstruct the embedding vector from index
            vec = index.reconstruct(idx).reshape(1, -1)
            sim = float(np.dot(q_norm[0], vec[0]))
            in_top40 = "YES" if idx in top40_set else "NO"
            name = expected_url.split('/view/')[-1].rstrip('/')
            print(f"{name:<50} {sim:<10.4f} {in_top40}")
            found = True
            break
    if not found:
        name = expected_url.split('/view/')[-1].rstrip('/')
        print(f"{name:<50} {'NOT IN INDEX':<10} {'N/A'}")